With the goal of reducing the fraction of negative weights in generated MC samples, we employ cell-reweighting [1] with an EMD [2] metric.

To perform cell-reweighting, negatively weighted events in a sample are chosen as the seeds of the cells. The size of the cell is increased to include surrounding events in phase space until the sum of the weights within the cell becomes positive. The weights of the events within the cell are then rescaled
$$ w_i\rightarrow \frac{\sum_{j\in C}w_j}{\sum_{j in C}|w_j|}|w_i| $$
for all events $i$ in cell $C$. This ensures that the cross-section of events is preserved globally.

The computational cost of cell-resampling is dominated by the nearest-neighbor search when forming the cells. A naive nearest-neighbors search scales like $\cal{O}(n^2)$. We employ a vantage-point tree based search like in [3]. To construct a vantage-point tree.... IDK.

Starting from numpy arrays of particle $(p_T,\eta,\phi)$ coordinates, we first make the vp tree.

## Running the workflow

The inputs are a `points` array of shape `(N_events, N_max_particles, 3)` holding `[pT, eta, phi]` per particle (zero-padded), and a `weights` array of length `N_events`. In the paper these are built from MadGraph/Pythia `.root` files by the scripts in `../scripts/preprocessing/`. To try the workflow without those samples, generate a small synthetic sample first:

```
python3 ../scripts/examples/make_toy_data.py --outdir ../data/toy_example --nevents 500
```

(`../scripts/examples/run_toy_example.sh` runs the whole chain below in one go.)

In [ ]:
import numpy as np
had_points = np.load('../data/toy_example/hadronization_points.npy')
weights = np.load('../data/toy_example/weights.npy')
print(had_points.shape, 'events x particles x [pT, eta, phi];', (weights < 0).sum(), 'negative weights')

In [ ]:
# build the vp-tree over the points (this is the expensive step for large samples)
!python3 -u ../scripts/reweighting/make_vptree.py --path=../data/vptree_pkls/toy_had_vptree.pkl --point_path=../data/toy_example/hadronization_points.npy --beta=1

In [ ]:
# reweight with a maximum cell radius of 20 GeV (EMD units)
!python3 -u ../scripts/reweighting/100k_cell_reweighting.py --vptree=../data/vptree_pkls/toy_had_vptree.pkl --points=../data/toy_example/hadronization_points.npy --weights=../data/toy_example/weights.npy --path=../data/toy_example/toy_had_emd_reweight_20gev.npy --max_radius=20 --whattype=2 --beta=1 --nproc=4

new_weights = np.load('../data/toy_example/toy_had_emd_reweight_20gev.npy')
print('negative weights before/after:', (weights < 0).sum(), (new_weights < 0).sum())
print('sum of weights before/after:', weights.sum(), new_weights.sum())